## Machine Learning Pipeline

As the class practice, the students will be required to develop a machine learning pipeline using the `Churn_Modelling_train_test.csv` dataset.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)


### Exploratory Data Analysis

In this section the students are required to do an EDA to understand the dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("Churn_Modelling_train_test.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nBasic Statistics:")
display(df.describe())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nTarget Distribution:")
print(df['Exited'].value_counts())
print(f"\nChurn Rate: {df['Exited'].mean():.2%}")

In [ ]:
categorical_cols = ['Geography', 'Gender']

for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts(dropna=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(categorical_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
binary_cols = ['HasCrCard', 'IsActiveMember']

for col in binary_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts(dropna=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(binary_cols):
    churn_rate = df.groupby(col)['Exited'].mean()
    churn_rate.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Churn Rate by {col}')
    axes[i].set_ylabel('Churn Rate')
    axes[i].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

print(df[numerical_cols].describe())

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    row, col_idx = divmod(i, 3)
    df.boxplot(column=col, by='Exited', ax=axes[row][col_idx])
    axes[row][col_idx].set_title(f'{col} by Exited')
    axes[row][col_idx].set_xlabel('Exited (0=Stayed, 1=Churned)')
plt.suptitle('')
plt.tight_layout()
plt.show()

**EDA Conclusions**

From this Exploratory Data Analysis we can retrieve the following information:
* The dataset has **9,001 rows and 14 columns**, with minor missing values in `Geography` (1), `Age` (1), and `HasCrCard` (1) — these rows will be dropped during preprocessing.
* The target variable `Exited` is **imbalanced**: ~79.6% of customers stayed and ~20.4% churned. Balancing the dataset before training is recommended.
* Among **categorical variables**: France accounts for ~50% of customers, Spain and Germany ~25% each. Gender is roughly balanced (~55% Male, ~45% Female).
* **Binary variables**: ~70% of customers have a credit card (`HasCrCard`). Active members (`IsActiveMember=1`) show a noticeably lower churn rate than inactive ones.
* For **numerical variables**: Churned customers tend to be **older** (higher `Age`). Customers with a **balance of 0** tend to stay. Customers with **3+ products** show very high churn rates — a key predictive signal.
* `RowNumber`, `CustomerId`, and `Surname` are identifiers with no predictive value and should be **dropped** before modeling.

### ML Pipeline

In this section the students are required to create a ML pipeline to predict whether the customer left the bank or not.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score
import joblib

In [ ]:
df = pd.read_csv("Churn_Modelling_train_test.csv")
print(df.shape)
df.head()

In [ ]:
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = ['RowNumber', 'CustomerId', 'Surname']
        self.ONE_HOT_ENCODE_COLUMNS = ['Geography']
        self.encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")

    def fit(self, df: pd.DataFrame) -> 'Transformer':
        df_clean = df.dropna()
        self.encoder.fit(df_clean[self.ONE_HOT_ENCODE_COLUMNS])
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.dropna().reset_index(drop=True)
        df = df.drop(columns=self.DROP_COLUMNS)
        df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
        encoded = self.encoder.transform(df[self.ONE_HOT_ENCODE_COLUMNS])
        df = df.drop(columns=self.ONE_HOT_ENCODE_COLUMNS)
        df = pd.concat([df, encoded], axis=1)
        return df

transformer = Transformer()
transformer.fit(df)
df_transformed = transformer.transform(df)
print("Transformed shape:", df_transformed.shape)
df_transformed.head()

In [ ]:
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df_0 = df[df['Exited'] == 0]
    df_1 = df[df['Exited'] == 1]
    min_size = len(df_1)
    df_0_balanced = df_0.sample(n=min_size, random_state=42)
    df_balanced = pd.concat([df_0_balanced, df_1])
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    return df_balanced

df_balanced = balance_dataset(df_transformed)
print("Balanced dataset shape:", df_balanced.shape)
print("Target distribution after balancing:")
print(df_balanced['Exited'].value_counts())

In [ ]:
X = df_balanced.drop('Exited', axis=1)
y = df_balanced['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "max_depth": 5,
    "min_samples_split": 10,
    "random_state": 42,
}

dt = DecisionTreeClassifier(**params)
dt.fit(X_train, y_train)
print("Model trained successfully!")
print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")

In [ ]:
y_pred = dt.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 Score:  {f1:.4f}")

### MLFlow Integration

In this section the students are required to track the model and the parameters into MLFlow.

In [ ]:
import mlflow
from mlflow.models import infer_signature

Start a local Tracking server on port 8080 - it is necessary to import mlflow and open a new terminal

In [ ]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

Visit the MLFlow UI on `http://127.0.0.1:8080`

In [ ]:
mlflow.set_experiment("Practice Experiment - Your Name")

In [ ]:
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("accuracy", accuracy)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Decision Tree model for Churn prediction")

    # Infer the model signature
    signature = infer_signature(X_train, dt.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=dt,
        artifact_path="churn_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="churn-model-decision-tree",
    )

print(f"Run logged successfully!")
print(f"Model URI: {model_info.model_uri}")

### Keep practicing

In [3]:
# Create a new experiment using other models or different parameters
...